# Тут представлены SQL запросы, написанные в рамках курса по изучения Баз Данных.

---

# Базовые запросы

## Одна БД

```
SELECT g.Group_Id, g.Name AS Group_Name, SUM(o.Quantity)
AS TotalQuantitySold
FROM Groups g
	JOIN Products p ON g.Group_Id = p.Group_Id
	JOIN Outgoing o ON p.Prod_Id = o.Prod_Id
GROUP BY
	g.Group_Id,
	g.Name;
```

```
-- 1.	Вывести  фамилию, имя, зачетку, группу, балл, год рождения  студентов 1 курса, 
-- отсортировав данные по группе по возрастанию, в пределах группы по убыванию года рождения.

SELECT s_name, f_name, n_z, n_gr, ball, EXTRACT(YEAR FROM data_b) year_b
WHERE FLOOR(n_gr / 1000) = 1
ORDER BY 4, 6 DESC

-- 2.	Вывести  названия хобби, риск и  тип для всех хобби, чей риск от m до n  
-- включая обе границы, отсортировать по убыванию риска.

SELECT h_name, risk, h_type
FROM public."Hobby"
WHERE risk >= 2 AND risk <= 9
ORDER BY risk DESC

-- 3.	Вывести  названия хобби, риск и  тип для всех хобби, чей риск от m до n  
-- включая обе границы, отсортировать по убыванию риска в пределах каждого типа хобби.

SELECT h_name, risk, h_type
FROM public."Hobby"
WHERE risk >= 4 AND risk <= 9
ORDER BY h_type, risk DESC

-- 4.	Вывести  фамилию, имя, зачетку, дату рождения студентов, чей балл от 3 до 4, 
-- включая только левую границу, имеющих день рождения в текущем месяце.

SELECT s_name, f_name, n_z, data_b
FROM public."Students"
WHERE ball >= 3 AND ball < 4 AND EXTRACT(MONTH FROM data_b) = EXTRACT(MONTH FROM CURRENT_DATE)

-- 5.	Вывести  названия тех хобби, которые действующие и месяц  начала 
-- увлечения  — любой летний месяц , отсортировав данные по месяцу по возрастанию. 

SELECT h_name
FROM public."Students_Hobby"
WHERE d_finish IS NULL AND EXTRACT(MONTH FROM d_start) IN (6, 7, 8)
ORDER BY EXTRACT(MONTH FROM d_start) ASC

-- 6.	Вывести  номер зачетки, название хобби и длительность в месяцах для всех завершенных хобби, 
-- отсортировав данные по зачетке по возрастанию, в пределах зачетки по длительности по убыванию.

SELECT n_z, h_name, EXTRACT(YEAR FROM -AGE(d_start, d_finish)) * 12 
	+ EXTRACT(MONTH FROM -AGE(d_start, d_finish)) duration
FROM public."Students_Hobby"
WHERE d_finish IS NOT NULL
ORDER BY 1, inter DESC

-- 7.	Вывести  фамилию, имя, зачетку, группу, месяц и год рождения всех отличников 2-го курса, 
-- отсортировав по группе во возрастанию, в пределах группы по возрастанию даты рождения.

SELECT s_name, f_name, n_z, n_gr, EXTRACT(MONTH FROM data_b) AS month_b, 
EXTRACT(YEAR FROM data_b) AS year_b
FROM public."Students"
WHERE ball >= 4.8
ORDER BY n_gr, data_b
```

```
-- 19. Вывести список менеджеров, проранжировав его по выручке текущего года.

SELECT Managers.Man_Id, Managers.Name, 
       SUM(Outgoing.Quantity * (Prices.Value * Courses.Value)) AS Revenue, 
       RANK() OVER (ORDER BY SUM(Outgoing.Quantity * (Prices.Value * Courses.Value)) DESC) AS Rank
FROM Managers
JOIN Outgoing ON Managers.Man_Id = Outgoing.Man_Id
JOIN Products ON Outgoing.Prod_Id = Products.Prod_Id
JOIN Prices ON Products.Prod_Id = Prices.Prod_Id AND Outgoing.Out_Date BETWEEN Prices.DayFrom AND Prices.DateTo
JOIN Courses ON Prices.Cur_Id = Courses.Cur_IdFrom
WHERE EXTRACT(YEAR FROM Outgoing.Out_Date) = EXTRACT(YEAR FROM CURRENT_DATE)
  AND Courses.DayFrom <= Outgoing.Out_Date AND (Courses.DayTo >= Outgoing.Out_Date OR Courses.DayTo IS NULL)
  AND Prices.DayFrom <= Outgoing.Out_Date AND (Prices.DateTo >= Outgoing.Out_Date OR Prices.DateTo IS NULL)
GROUP BY 1, 2
ORDER BY 3 DESC;
```

---

## Несколько БД

```
-- 1.	Вывести  фамилию, имя, зачетку, дату рождения, название хобби и длительность в месяцах для всех завершенных хобби.

SELECT s_name, f_name, S.n_z, data_b, SH.h_name, EXTRACT(YEAR FROM -AGE(d_start, d_finish)) * 12 
	+ EXTRACT(MONTH FROM -AGE(d_start, d_finish)) duration
FROM public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND d_finish IS NOT NULL

-- 2.	Вывести  для каждого названия хобби риск, тип и номера зачеток студентов, которые занимаются им на текущую дату.

SELECT H.h_name, risk, h_type, n_z
FROM public."Hobby" H, public."Students_Hobby" SH
WHERE H.h_name = SH.h_name AND d_finish IS NULL

-- 3.	Вывести  фамилию, имя, зачетку, дату рождения студентов, чей балл от 4 до 5 включительно, 
-- чей день рождения в текущем или следующем за текущим месяце и которые имеют законченные хобби .

SELECT s_name, f_name, S.n_z, data_b
FROM public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND ball >= 4 AND ball <= 5
	AND (EXTRACT(MONTH FROM data_b) = EXTRACT(MONTH FROM CURRENT_DATE) 
	 OR EXTRACT(MONTH FROM data_b) = (EXTRACT(MONTH FROM CURRENT_DATE) + 1)) 
	AND d_finish IS NOT NULL

-- 4.	Вывести  фамилию, имя, зачетку, дату рождения студентов, название законченных хобби для студентов, 
-- которым исполнилось N  полных лет на  текущую дату и день рождения которых в текущем месяце.

SELECT s_name, f_name, S.n_z, data_b, h_name
FROM public."Students" S, public."Students_Hobby" SH
WHERE
	S.n_z = SH.n_z AND
	EXTRACT(MONTH FROM data_b) = EXTRACT(MONTH FROM CURRENT_DATE) AND
	EXTRACT(YEAR FROM -AGE(data_b, current_date)) >= 18 AND
	d_finish IS NOT NULL

-- 5.	Вывести  фамилию, имя, зачетку, дату рождения, номер курса для всех отличников,  не имеющих хобби, 
-- отсортировав данные по курсу по возрастанию, в пределах курса по убыванию даты рождения.

SELECT s_name, f_name, n_z, data_b, FLOOR(n_gr/1000) course
FROM public."Students"
WHERE n_z NOT IN (SELECT n_z FROM public."Students_Hobby") AND ball >= 4.8
ORDER BY course, data_b DESC

-- 6.	Вывести  фамилию, имя, зачетку, количество полных лет для каждого из студентов 2-х заданных групп, 
-- увлекающихся на текущую дату заданным типом хобби, отсортировав данные по группе по возрастанию, в пределах группы по возрастанию возраста.

SELECT s_name, f_name, S.n_z, EXTRACT(YEAR FROM -AGE(data_b, current_date)) old
FROM public."Students" S, public."Hobby" H, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND H.h_name = SH.h_name AND n_gr IN (2011, 2012) AND d_finish IS NULL
	AND h_type = 'спорт'
ORDER BY n_gr, old

-- 7.	Вывести  названия и риск тех хобби, которыми увлекаются только отличники 1,2 или 3 курсов, отсортировав данные по убыванию риска. 

SELECT H.h_name, risk
FROM public."Students" S, public."Hobby" H, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND H.h_name = SH.h_name AND ball >= 4.8 AND FLOOR(n_gr/1000) IN (1, 2, 3)
ORDER BY risk DESC
```

---

## Виртуальная таблица

```
-- 1.	Найти номера групп с минимальным количеством троечников.
	
SELECT S.n_gr
FROM public."Students" S
GROUP BY S.n_gr
HAVING COUNT(CASE WHEN S.ball < 4 THEN 1 END) = (
  SELECT COUNT(CASE WHEN ball < 4 THEN 1 END) AS count_bad
  FROM public."Students"
  GROUP BY n_gr
  ORDER BY count_bad
  LIMIT 1
);
	
-- 2.	Вывести  фамилию, имя, зачетку, дату рождения студента (ов), имеющего максимальное количество хобби заданного типа.

SELECT s_name, f_name, S.n_z, data_b
FROM public."Students" S, public."Hobby" H, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND H.h_name = SH.h_name AND h_type = 'спорт'
GROUP BY 3
HAVING COUNT(*) = (
	SELECT COUNT(*) 
	FROM public."Students_Hobby" SH2, public."Hobby" H2
	WHERE SH2.h_name = H2.h_name AND S.n_z = SH2.n_z AND H2.h_type = 'спорт'
	GROUP BY SH2.n_z 
	ORDER BY COUNT(*) DESC 
	LIMIT 1 
);
	
-- 3.	Найти номера групп с максимальным количеством отличников.

SELECT n_gr 
FROM public."Students"
GROUP BY 1
HAVING COUNT(CASE WHEN ball = 5 THEN 1 END) = (
	SELECT COUNT(CASE WHEN ball = 5 THEN 1 END) AS count_five
	FROM public."Students"
	GROUP BY n_gr 
	ORDER BY COUNT(*) DESC 
	LIMIT 1 
);
	
-- 4.	Найти самое популярное хобби среди всех студентов.

SELECT H.h_name 
FROM public."Hobby" H, public."Students_Hobby" SH
WHERE H.h_name = SH.h_name 
GROUP BY 1
HAVING COUNT(*) = (
	SELECT COUNT(*) 
	FROM public."Students_Hobby" SH2
	WHERE H.h_name = SH2.h_name 
	GROUP BY SH2.h_name
	ORDER BY COUNT(*) DESC 
	LIMIT 1 
);
	
-- 5.	Вывести  название хобби, риск  и количество студентов, которые им увлекаются на текущую дату 
-- для самого популярного хобби среди студентов 2-го курса.

SELECT H.h_name, risk, COUNT(SH.n_z) AS num_students
FROM public."Students_Hobby" SH
INNER JOIN public."Hobby" H ON SH.h_name = H.h_name
INNER JOIN public."Students" S ON SH.n_z = S.n_z
WHERE FLOOR(n_gr/1000) = 2 AND d_finish IS NULL
GROUP BY 1, 2
HAVING COUNT(SH.n_z) = (
  SELECT COUNT(SH2.n_z)
  FROM public."Students_Hobby" SH2
  INNER JOIN public."Students" S2 ON SH2.n_z = S2.n_z
  WHERE FLOOR(n_gr/1000) = 2 AND d_finish IS NULL
  GROUP BY SH2.h_name
  ORDER BY COUNT(SH2.n_z) DESC
  LIMIT 1
)
ORDER BY num_students DESC

-- 6.	Найти самое популярное хобби на каждом курсе.

CREATE OR REPLACE VIEW V1 AS
SELECT n_gr, h_name, COUNT(h_name) AS amount
FROM public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z
GROUP BY 1, 2;

CREATE OR REPLACE VIEW V2 AS
SELECT n_gr, MAX(V1.amount)
FROM V1
GROUP BY 1;

SELECT V1.*
FROM V1, V2
WHERE V1.n_gr = V2.n_gr AND V1.amount = V2.max


-- 7.	Вывести  фамилию, имя, зачетку, дату рождения  самого молодого из студентов, имеющих более одного действующего хобби.

SELECT S.s_name, S.f_name, S.n_z, S.data_b
FROM public."Students" S
INNER JOIN (
  SELECT SH.n_z, COUNT(*) AS num_hobbies
  FROM public."Students_Hobby" SH
  WHERE SH.d_finish IS NULL
  GROUP BY 1
  HAVING COUNT(*) > 1
) SH2 ON S.n_z = SH2.n_z
WHERE S.data_b = (
  SELECT MIN(S2.data_b)
  FROM public."Students" S2
  INNER JOIN public."Students_Hobby" SH3 ON S2.n_z = SH3.n_z
  WHERE S2.n_z = S.n_z AND SH3.d_finish IS NULL
)

-- 8.	Найти самого молодого студента, который учится в самой многочисленной группе.

SELECT *
FROM public."Students" S
WHERE n_gr IN (
	SELECT n_gr
	FROM public."Students"
	GROUP BY 1
	HAVING COUNT(*) = (
		SELECT MAX(count_num)
		FROM (
			SELECT COUNT(*) AS count_num
			FROM public."Students"
			GROUP BY n_gr) V)
	) AND data_b = (
		SELECT MAX(S1.data_b)
		FROM public."Students" S1
		WHERE S1.n_gr = S.n_gr);

-- 9.	Найти хобби с максимальным риском среди самых популярных хобби среди студентов 2-го курса.

SELECT H.h_name, H.risk AS max_risk
FROM public."Hobby" H
INNER JOIN (
  SELECT h_name
  FROM public."Students_Hobby" SH2
  WHERE n_z IN (SELECT n_z FROM public."Students" WHERE FLOOR(n_gr/1000) = 2)
  GROUP BY 1
  ORDER BY COUNT(*) DESC
  LIMIT 5
) SH ON H.h_name = SH.h_name
WHERE H.risk = (
  SELECT MAX(risk)
  FROM public."Hobby" H3
  WHERE h_name IN (
    SELECT h_name
	FROM public."Students_Hobby" SH3
    WHERE n_z IN (SELECT n_z FROM public."Students" WHERE FLOOR(n_gr/1000) = 2)
    GROUP BY 1
	ORDER BY COUNT(*) DESC
	LIMIT 5
	)
);

-- 10.	Для каждой группы подсчитать отдельно количество всех студентов в группе, количество троечников и количество отличников.

SELECT n_gr, COUNT(*) AS count_students, 
       SUM(CASE WHEN ball < 4 THEN 1 ELSE 0 END) AS num_threes, 
       SUM(CASE WHEN ball = 5 THEN 1 ELSE 0 END) AS num_fives
FROM public."Students"
GROUP BY 1;

-- 11.	Для каждого курса подсчитать отдельно количество всех студентов на курсе и  количество отличников.

SELECT FLOOR(n_gr/1000) AS num_course, COUNT(*) AS count_students, 
       SUM(CASE WHEN ball = 5 THEN 1 ELSE 0 END) AS num_fives
FROM public."Students"
GROUP BY 1;
```

---

## Групповые функции

```
-- 1.	Вывести  для каждого названия хобби риск и количество различных групп, в которых увлекаются этим хобби на текущую дату.

SELECT H.h_name, risk, COUNT(DISTINCT n_gr) AS gr_count
FROM public."Hobby" H,
	(SELECT H.h_name, n_gr 
	 FROM public."Hobby" H, public."Students" S, public."Students_Hobby" SH
	 WHERE H.h_name = SH.h_name AND S.n_z = SH.n_z AND SH.d_finish IS NULL) Z1
WHERE H.h_name = Z1.h_name
GROUP BY 1

-- 2.	Вывести  фамилию, имя, зачетку, дату рождения студентов, которым исполнилось N  полных лет на  текущую дату, 
-- и которые имеют более 1 действующего хобби.

SELECT s_name, f_name, S.n_z, data_b
FROM public."Students" S, 
	(SELECT S.n_z, COUNT(DISTINCT SH.n_z) AS count
	 FROM public."Students_Hobby" SH, public."Students" S
	 WHERE S.n_z = SH.n_z
	 GROUP BY 1) Z1
WHERE S.n_z = Z1.n_z AND EXTRACT(YEAR FROM -AGE(data_b, current_date)) >= 17 AND Z1.count > 1

-- 3.	Найти все хобби (названия хобби и риск), которыми увлекаются студенты, имеющие максимальный балл.

SELECT H.h_name, risk
FROM public."Hobby" H, public."Students_Hobby" SH, public."Students" S
WHERE S.n_z = SH.n_z AND H.h_name = SH.h_name AND S.ball = (SELECT MAX(ball) FROM public."Students")
GROUP BY 1

-- 4.	Найти средний балл в каждой группе, учитывая только баллы студентов, 
-- имеющих хотя бы 1 действующее хобби  заданного типа.

SELECT n_gr, AVG(ball) AS avg_ball
FROM public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND h_name IN (SELECT h_name FROM public."Hobby" WHERE h_type = 'спорт' ) 
	AND d_finish IS NOT NULL
GROUP BY 1

-- 5.	Найти номера групп, в которых есть отличники, увлекающиеся любым спортом не менее N месяцев от текущей даты.

SELECT DISTINCT n_gr
FROM public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND ball >= 4.8 
	AND h_name IN (SELECT h_name FROM public."Hobby" WHERE h_type = 'спорт' )
	AND d_finish IS NOT NULL 
	AND EXTRACT(YEAR FROM -AGE(d_start, CURRENT_DATE)) * 12 + 
	EXTRACT(MONTH FROM -AGE(d_start, CURRENT_DATE)) >= 8
	
-- 6.	Найти название, тип, риск, длительность в месяцах самого продолжительного хобби из действующих, 
-- указав номер зачетки  студента и номер его группы.

SELECT H.h_name, h_type, risk, EXTRACT(YEAR FROM -AGE(d_start, CURRENT_DATE)) * 12 + 
	EXTRACT(MONTH FROM -AGE(d_start, CURRENT_DATE)) AS duration, S.n_z, n_gr
FROM public."Hobby" H, public."Students" S, public."Students_Hobby" SH
WHERE H.h_name = SH.h_name AND S.n_z = SH.n_z AND d_finish IS NULL
ORDER BY duration DESC
LIMIT 1

-- 7.	Найти всех студентов, чей балл выше среднего, отсортировав по номеру группы по возрастанию, 
-- в пределах группы по убыванию балла.

SELECT * FROM public."Students"
WHERE ball > (SELECT AVG(ball) FROM public."Students")
ORDER BY n_gr, ball DESC

-- 8.	Найти все действующие хобби, которыми увлекаются троечники 2-го курса.

SELECT * FROM public."Hobby" H, public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND H.h_name = SH.h_name AND ball < 4 AND FLOOR(n_gr / 1000) = 2 
	AND  d_finish IS NULL
	
-- 9.	Найти номера курсов, на которых студенты имеют в среднем более одного действующего хобби.

SELECT FLOOR(n_gr/1000) AS n_course
FROM public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND d_finish IS NULL
GROUP BY n_course
HAVING COUNT(*) > 1

/*
SELECT Z1.n_k
FROM (SELECT DIV(n_gr, 1000) AS n_k, COUNT(n_z) AS z_count FROM public."Students" S GROUP BY 1) Z1,
	(SELECT DIV(n_gr, 1000) AS n_k, COUNT(h_name) AS h_count 
	 FROM public."Students" S, public."Students_Hobby" SH 
	 WHERE S.n_z = SH.n_z AND d_finish IS NULL GROUP BY 1) Z2
WHERE Z1.n_k = Z2.n_k AND Z1.z_count < Z2.h_count
*/

-- 10.	Найти номера групп, в которых не менее 50% студентов имеют балл не ниже 4.

SELECT n_gr
FROM public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z
GROUP BY n_gr
HAVING COUNT(CASE WHEN ball >= 4 THEN 1 ELSE NULL END) >= COUNT(*)/2

/*
SELECT Z1.n_gr
FROM (SELECT n_gr, COUNT(n_z) AS count_ball FROM public."Students" SAMPLE
	WHERE ball >= 4 GROUP BY 1) Z1,
	(SELECT n_gr, COUNT(n_z) AS count_z FROM public."Students" S GROUP BY 1) Z2
WHERE Z1.count_ball / Z2.count_z > 0.5 AND Z1.n_gr = Z2.n_gr
*/

-- 11.	Для каждого курса подсчитать количество различных  действующих хобби на курсе.

SELECT FLOOR(n_gr/1000) AS n_course, COUNT(DISTINCT H.h_name) AS count_hobby
FROM public."Hobby" H, public."Students" S, public."Students_Hobby" SH
WHERE S.n_z = SH.n_z AND H.h_name = SH.h_name AND d_finish IS NULL
GROUP BY n_course 

-- 12.	Вывести  фамилию, имя, зачетку, группу, количество  всех действующих хобби для тех студентов, 
-- которые имеют максимальных балл в своей группе.

SELECT s_name, f_name, S.n_z, n_gr, COUNT(h_name)
FROM public."Students_Hobby" SH, public."Students" S
WHERE S.n_z = SH.n_z AND ball = (SELECT MAX(ball) FROM public."Students" WHERE n_gr = S.n_gr) AND d_finish IS NULL
GROUP BY 1, 2, 3, 4
```

---

# Запросы для более сложной БД

```
Таблица Dealers, в которой есть следующие поля: D_Id (primary key), Name, Percent, Comments.
Таблица Managers с следующими полями: Man_Id (PK), D_Id (FK), Name, Percent, Hire_Day. Comments, Parent_Id.
Таблица Contragents с следующими полями: Contr_Id (PK), Name, Address, Phone, Comments.
Таблица Contracts со следующими полями: Contr_Id (FK), Man_Id (FK), DayFrom, DayTo.
Таблица Groups со следующими полями: Group_Id (PK), Name, Comments.
Таблица Products со следующими полями: Prod_Id (PK), Group_Id (FK), Name, Description, Expire_Time.
Таблица Currencies со следующими полями: Cur_Id (PK), Name, ShortName. 
Таблица Prices со следующими полями: Prod_Id (FK), DayFrom, Cur_Id (FK), DateTo, Value. Prod_Id и DayFrom - primary key.
Таблица Courses со следующими полями: Cur_IdFrom (FK), Cur_IdTo (FK), DayFrom, DayTo, Value.
Таблица Taxes со следующими полями: Tax_Id (PK), Name, Value, Comments
Таблица Incoming со следующими полями: Inc_Id (PK), Prod_Id (FK), Tax_Id (FK), Contr_Id (FK), Man_Id (FK), Inc_Date, Quantity, Cost.
Таблица Outgoing со следующими полями: Out_Id (PK), Prod_Id (FK), Tax_Id (FK), Contr_Id (FK), Man_Id (FK), Out_Date, Quantity, Cost.
Таблица Banks со следующими полями: Bank_Id (PK), Name, Address.
Таблица Accounts со следующими полями: Acc_Id (PK), Bank_Id (FK), Cont_Id (FK), AccountNumber, DayFrom, DayTo.
```

---

```
-- 1. Вывести список товаров: Id, название, название группы товаров, стоимость в рублях на единицу актуальных на текущую дату, 
-- для всех товаров поступивших в текущем квартале.

SELECT Products.Prod_Id, Products.Name, Groups.Name AS GroupName, 
       (Prices.Value * Courses.Value) AS PriceInRubles
FROM Products
JOIN Groups ON Products.Group_Id = Groups.Group_Id
JOIN Prices ON Products.Prod_Id = Prices.Prod_Id AND CURRENT_DATE BETWEEN Prices.DayFrom AND Prices.DateTo
JOIN Courses ON Prices.Cur_Id = Courses.Cur_IdFrom
JOIN Incoming ON Products.Prod_Id = Incoming.Prod_Id
WHERE EXTRACT(QUARTER FROM Incoming.Inc_Date) = EXTRACT(QUARTER FROM CURRENT_DATE)

-- 2. Для каждого менеджера вывести Id, его стаж в годах, количество заключённых контрактов, суммарную выручку, 
-- за последние три месяца от текущей даты.

SELECT Managers.Man_Id,
       DATE_PART('year', AGE(CURRENT_DATE, Managers.Hire_Day)) AS Experience,
       COUNT(Contracts.Contr_Id) AS ContractCount,
       SUM(Outgoing.Quantity * (Prices.Value * Courses.Value)) AS Revenue
FROM Managers
JOIN Contracts ON Managers.Man_Id = Contracts.Man_Id
JOIN Outgoing ON Contracts.Contr_Id = Outgoing.Contr_Id
JOIN Products ON Outgoing.Prod_Id = Products.Prod_Id
JOIN Prices ON Products.Prod_Id = Prices.Prod_Id AND Outgoing.Out_Date BETWEEN Prices.DayFrom AND Prices.DateTo
JOIN Currencies AS CurrencyTo ON Prices.Cur_Id = CurrencyTo.Cur_Id
JOIN Courses ON CurrencyTo.Cur_Id = Courses.Cur_IdTo AND Outgoing.Out_Date BETWEEN Courses.DayFrom AND Courses.DayTo
WHERE Outgoing.Out_Date BETWEEN CURRENT_DATE - INTERVAL '3 months' AND CURRENT_DATE
GROUP BY Managers.Man_Id, Managers.Hire_Day

-- 3. Для каждого контрагента вывести количество действующих и количество завершённых контрактов за последний год.

SELECT c.Contr_Id, 
    SUM(CASE WHEN c.DayTo IS NULL THEN 1 END) AS ActiveContracts, 
    SUM(CASE WHEN c.DayTo IS NOT NULL THEN 1 END) AS CompletedContracts
FROM Contragents a
JOIN Contracts c ON a.Contr_Id = c.Contr_Id
WHERE c.DayFrom >= CURRENT_DATE - INTERVAL '1 year'
GROUP BY c.Contr_Id;

-- 4. Вывести список товаров имеющихся в наличии.

(
	SELECT inc.Prod_Id
	FROM Incoming inc
	LEFT JOIN Outgoing og ON inc.Prod_Id = og.Prod_Id
	GROUP BY 1
	HAVING SUM(inc.Quanity) < SUM(og.Quanity)
) S1
UNION
(
	SELECT Prod_Id
	FROM Incoming 
	WHERE 1 NOT IN (SELECT Prod_Id FROM Outgoing)
) S2;

-- 5. Вывести наиболее часто продаваемые товары в каждой группе.

SELECT Group_Id, Prod_Id
FROM (
	SELECT Group_Id, MAX()
	FROM (
		SELECT Group_Id, og.Prod_Id, COUNT(og.Prod_Id) 
		FROM Outgoing og
		JOIN Products pr ON og.Prod_Id = pr.Prod_Id
		GROUP BY 1, 2
	) Z1
	GROUP BY 1
) Z2
```

---

## Функции PL/pgSQL

```
-- 2 Написать код с обработкой исключений, скомпилировать и проверить на тестовых
-- данных следующие функции и процедуры PL/SQL:

-- 1) функция, возвращающая наименование группы продукта по его идентификатору и обратную ей функцию;


-- Функция, возвращающая наименование группы продукта по его идентификатору
CREATE OR REPLACE FUNCTION getGroupNameById(product_id INT)
RETURNS TEXT AS $$
DECLARE
    group_name TEXT;
BEGIN
    SELECT g.Name INTO group_name
    FROM Products p
    JOIN Groups g ON p.Group_Id = g.Group_Id
    WHERE p.Prod_Id = product_id;

    IF group_name IS NULL THEN
        RAISE EXCEPTION 'Продукт с указанным идентификатором не найден: %', product_id;
    END IF;

    RETURN group_name;
	
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RAISE NOTICE 'Нет данных для обработки.';
    WHEN OTHERS THEN
        RAISE EXCEPTION 'Ошибка в курсоре: %', SQLERRM;
END;
$$ LANGUAGE PLPGSQL;

-- Обратная функция, возвращающая идентификатор группы продукта по его наименованию
CREATE OR REPLACE FUNCTION getGroupIdByName(product_name TEXT)
RETURNS SETOF INT AS $$
DECLARE
    group_id INT;
BEGIN
	FOR group_id IN
		SELECT p.Group_Id
		FROM Groups g
		JOIN Products p ON g.Group_Id = p.Group_Id
		WHERE LOWER(p.Name) = LOWER(product_name)
	LOOP
		RETURN NEXT group_id;
	END LOOP;
	
    IF NOT FOUND THEN
        RAISE EXCEPTION 'Группа продукта с указанным наименованием не найдена: %', product_name;
    END IF;
	
EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RAISE NOTICE 'Нет данных для обработки.';
    WHEN OTHERS THEN
        RAISE EXCEPTION 'Ошибка в курсоре: %', SQLERRM;
END;
$$ LANGUAGE PLPGSQL;



-- 2) функция, возвращающая средний объем поступления или продаж (параметр) указанной
-- названием группы продуктов за месяц (указанный в формате «ММ.ГГГГ»);

CREATE OR REPLACE FUNCTION getAverageVolumeByGroupName(
    group_name TEXT,
    month_year TEXT,
    operation_type TEXT
)
RETURNS NUMERIC AS $$
DECLARE
    average_volume NUMERIC;
    date_field TEXT;
BEGIN
    -- Проверка корректности введенного месяца
    IF NOT regexp_match(month_year, '^\d{2}\.\d{4}$') THEN
        RAISE EXCEPTION 'Некорректный формат месяца (ожидается "ММ.ГГГГ"): %', month_year;
    END IF;

    -- Извлечение месяца и года из формата "ММ.ГГГГ"
    DECLARE
        month INT := CAST(SUBSTRING(month_year, 1, 2) AS INT);
        year INT := CAST(SUBSTRING(month_year, 4, 4) AS INT);
    BEGIN
        -- Определение имени поля даты в зависимости от operation_type
        IF LOWER(operation_type) = 'поступление' THEN
            date_field := 'inc_date';
        ELSIF LOWER(operation_type) = 'продажа' THEN
            date_field := 'out_date';
        ELSE
            RAISE EXCEPTION 'Некорректный тип операции: %', operation_type;
        END IF;

        -- Вычисление среднего объема по указанной группе продуктов и месяцу
        EXECUTE format('
            SELECT AVG(volume) FROM (
                SELECT
                    SUM(Quantity) AS volume
                FROM
                    Products p
                    JOIN Groups g ON p.Group_Id = g.Group_Id
                    JOIN %I i ON p.Prod_Id = i.Prod_Id
                WHERE
                    LOWER(g.Name) = LOWER(%L)
                    AND EXTRACT(MONTH FROM %I) = %s
                    AND EXTRACT(YEAR FROM %I) = %s
                GROUP BY
                    EXTRACT(MONTH FROM %I), EXTRACT(YEAR FROM %I)
            ) subquery',
            operation_type, group_name, date_field, month, date_field, year, date_field
        ) INTO average_volume;

        RETURN COALESCE(average_volume, 0);
		
    EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RAISE NOTICE 'Нет данных для обработки.';
    WHEN OTHERS THEN
        RAISE EXCEPTION 'Ошибка в функции: %', SQLERRM;
    END;
END;
$$ LANGUAGE PLPGSQL;


-- 3) Используя пользовательские функции напишите процедуру отображающую средний
-- объем поступления и продажи указанной группы продукта по всем месяцам текущего года.

CREATE OR REPLACE PROCEDURE displayAverageVolumesByGroup(
    group_name TEXT
) AS $$
DECLARE
    current_month INT;
    current_year INT;
    month_year TEXT;
BEGIN
    SELECT EXTRACT(MONTH FROM current_date) INTO current_month;
    SELECT EXTRACT(YEAR FROM current_date) INTO current_year;

    -- Временная таблица для хранения результатов
    CREATE TEMP TABLE temp_results (
        month_year TEXT,
        avg_volume_postuplenie NUMERIC,
        avg_volume_prodazha NUMERIC
    );

    FOR month_num IN 1..current_month LOOP
        -- Преобразование в формат "ММ.ГГГГ"
        IF month_num < 10 THEN
            month_year := '0' || month_num || '.' || current_year;
        ELSE
            month_year := month_num || '.' || current_year;
        END IF;

        -- Средний объем поступления и продаж в текущем месяце
        INSERT INTO temp_results (month_year, avg_volume_postuplenie, avg_volume_prodazha)
        VALUES (
            month_year,
            getAverageVolumeByGroupName(group_name, month_year, 'поступление'),
            getAverageVolumeByGroupName(group_name, month_year, 'продажа')
        );
    END LOOP;

    SELECT * FROM temp_results;
    DROP TABLE temp_results;
END;
$$ LANGUAGE PLPGSQL;
```

---

## Триггеры

```
-- Триггер, реагирующий на изменение цены товара в таблице Prices. Он должен отклонять операцию, если скидка будет более 25%.

-- Без WHEN

CREATE OR REPLACE FUNCTION checkPriceDiscount()
RETURNS TRIGGER AS $$
BEGIN	
    IF NEW.Value < OLD.Value * 0.75 THEN
        RAISE NOTICE 'Скидка превышает % %%. Изменение цены отклонено.', discount_percent;
    END IF;

    RETURN OLD;
END;
$$ LANGUAGE PLPGSQL;

CREATE TRIGGER before_update_price
BEFORE UPDATE ON Prices
FOR EACH ROW
EXECUTE FUNCTION checkPriceDiscount();


-- С использованием WHEN

CREATE OR REPLACE FUNCTION checkPriceDiscount()
RETURNS TRIGGER AS $$
BEGIN
    RAISE EXCEPTION 'Скидка превышает 25%. Изменение цены отклонено.', ;
	
    RETURN OLD;
END;
$$ LANGUAGE PLPGSQL;

CREATE TRIGGER before_update_price
BEFORE UPDATE ON Prices
FOR EACH ROW
WHEN (NEW.Value < OLD.Value * 0.75)
EXECUTE FUNCTION checkPriceDiscount();



-- 1) Запретить удаление товара из Products, если он есть в наличии.

CREATE OR REPLACE FUNCTION preventProductDeletion()
RETURNS TRIGGER AS $$
DECLARE
    incoming_quantity INT;
    outgoing_quantity INT;
BEGIN
    -- Количество поступлений для данного продукта
    SELECT COALESCE(SUM(Quantity), 0) INTO incoming_quantity
    FROM Incoming
    WHERE Prod_Id = OLD.Prod_Id;

    -- Количество продаж для данного продукта
    SELECT COALESCE(SUM(Quantity), 0) INTO outgoing_quantity
    FROM Outgoing
    WHERE Prod_Id = OLD.Prod_Id;

    IF incoming_quantity > outgoing_quantity THEN
        RAISE EXCEPTION 'Нельзя удалить товар, так как он есть в наличии';
    END IF;

    RETURN OLD;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER prevent_product_deletion
BEFORE DELETE ON Products
FOR EACH ROW
EXECUTE FUNCTION preventProductDeletion();



-- 2) Отслеживать все изменения данных в таблице Prices, записывая в таблицу
-- ARCHIVE пользователя, который произвел изменения, дату и время изменения,
-- действие (Insert/Update/Delete) и количество измененных строк.

CREATE OR REPLACE FUNCTION prices_audit()
RETURNS TRIGGER AS $$
BEGIN
    IF TG_OP = 'UPDATE' THEN
        -- Операция UPDATE
        INSERT INTO Archive (user_id, change_time, action, row_count)
        VALUES (pg_backend_pid(), current_timestamp, TG_OP, 1)
        ON CONFLICT (user_id, change_time, action) DO UPDATE
        SET row_count = Archive.row_count + 1
        WHERE Archive.user_id = pg_backend_pid() AND Archive.action = 'Update';
    ELSIF TG_OP = 'INSERT' THEN
        -- Операция INSERT
        INSERT INTO Archive (user_id, change_time, action, row_count)
        VALUES (pg_backend_pid(), current_timestamp, TG_OP, 1)
        ON CONFLICT (user_id, change_time, action) DO UPDATE
        SET row_count = Archive.row_count + 1
        WHERE Archive.user_id = pg_backend_pid() AND Archive.action = 'Insert';
    ELSIF TG_OP = 'DELETE' THEN
        -- Операция DELETE
        INSERT INTO Archive (user_id, change_time, action, row_count)
        VALUES (pg_backend_pid(), current_timestamp, TG_OP, 1)
        ON CONFLICT (user_id, change_time, action) DO UPDATE
        SET row_count = Archive.row_count + 1
        WHERE Archive.user_id = pg_backend_pid() AND Archive.action = 'Delete';
    END IF;
    RETURN NULL;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER prices_audit_trigger
AFTER INSERT OR UPDATE OR DELETE ON Prices
FOR EACH ROW
EXECUTE FUNCTION prices_audit();
```

---

## Замещающий триггер

```
-- 1. Создать представление, содержащее id продажи, id товара, имя товара, цену в рублях на дату продажи, количество, 
-- имя менеджера, дату продажи.

CREATE OR REPLACE VIEW SalesView AS
SELECT
    o.Out_Id AS sale_id,
    o.Prod_Id AS product_id,
    o.Contr_Id AS contragent_id,
    p.Name AS product_name,
    o.Quantity * (pr.Value * c.Value + (pr.Value * c.Value * t.Value / 100)) AS price_rub,
    o.Quantity,
    m.Name AS manager_name,
    o.Out_Date AS sale_date
FROM
    Outgoing o
JOIN
    Products p ON o.Prod_Id = p.Prod_Id
JOIN 
	Prices pr ON p.Prod_Id = pr.Prod_Id AND (o.Out_Date BETWEEN pr.DayFrom 
	AND pr.DateTo OR o.Out_Date >= pr.DayFrom AND pr.DateTo IS NULL)
JOIN
    Managers m ON o.Man_Id = m.Man_Id
JOIN
    Currencies cur ON pr.Cur_Id = cur.Cur_Id
JOIN 
	Courses c ON cur.Cur_Id = c.Cur_IdFrom AND (o.Out_Date BETWEEN c.DayFrom AND c.DayTo 
	OR o.Out_Date >= c.DayFrom AND c.DayTo IS NULL)
JOIN
    Taxes t ON o.Tax_Id = t.Tax_Id;


-- 2. Создать  замещающий триггер, который будет разрешать продавать только те товары, которые есть в наличии.

CREATE OR REPLACE FUNCTION check_product_availability_insert()
RETURNS TRIGGER AS $$
BEGIN
    -- Проверяем наличие товара перед вставкой
    IF NEW.Quantity <= (SELECT COALESCE(SUM(Quantity), 0) FROM Incoming WHERE Prod_Id = NEW.product_id) THEN
        -- Если товар есть в наличии, выполняем вставку
        INSERT INTO Outgoing (Prod_Id, Contr_Id, Man_Id, Tax_Id, Quantity, Out_Date)
        VALUES (NEW.product_id, NEW.contragent_id, NEW.man_id, NEW.tax_id, NEW.quantity, NEW.sale_date);
    ELSE
        -- Если товара нет в наличии, выбрасываем исключение
        RAISE EXCEPTION 'Продукта недостаточно в наличии для продажи';
    END IF;
    RETURN NULL;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER check_product_availability_insert_trigger
INSTEAD OF INSERT ON SalesView
FOR EACH ROW
EXECUTE FUNCTION check_product_availability_insert();


-- 3. Создать замещающий триггер, который будет разрешать удалять записи о продажах товаров, проданных более, 
-- чем 2 года назад от текущей даты.

CREATE OR REPLACE FUNCTION allow_sale_deletion_old_sales()
RETURNS TRIGGER AS $$
BEGIN
    -- Проверяем, была ли продажа более 2 лет назад
    IF EXTRACT(YEAR FROM AGE(current_date, OLD.sale_date)) >= 2 THEN
        DELETE FROM Outgoing WHERE Out_Id = OLD.sale_id;
    ELSE
        RAISE EXCEPTION 'Нельзя удалять записи продажи товаров, проданных менее 2 лет назад';
    END IF;
	RETURN NULL;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER allow_sale_deletion_old_sales_trigger
INSTEAD OF DELETE ON SalesView
FOR EACH ROW
EXECUTE FUNCTION allow_sale_deletion_old_sales();
```